# পাঠ ১৮: ক্রিপ্টোগ্রাফিক রসিদ দ্বারা এআই এজেন্ট সুরক্ষা

## হাতেকলমে ডায়েরী

এই ডায়েরীর মাধ্যমে চারটি অনুশীলন করা হবে:

১. একটি এজেন্ট টুল কলের জন্য **আপনার প্রথম রসিদে স্বাক্ষর করুন** এবং এটি যাচাই করুন।
২. রসিদে **পরিচ্ছেদ করুন** এবং যাচাই ব্যর্থ হওয়া দেখুন।
৩. একটি তিন-রসিদের চেন তৈরি করুন এবং চেইনের অখণ্ডতা নিশ্চিত করুন।
৪. একটি Microsoft Agent Framework টুল কল **র‍্যাপ করুন** যাতে প্রতিটি অ্যাকশন একটি রসিদ তৈরি করে।

সব ধরনের ক্রিপ্টোগ্রাফিক প্রিমিটিভ ভালভাবে পরিচালিত লাইব্রেরি থেকে আমদানি করা হয়েছে (`pynacl` Ed25519 এর জন্য, `jcs` RFC 8785 canonical JSON এর জন্য, এবং পাইটনের স্ট্যান্ডার্ড লাইব্রেরি `hashlib` SHA-256 এর জন্য)। রসিদ লজিকটি নিজেই সাদামাটা পাইথন, যা আপনি পড়তে এবং পরিবর্তন করতে পারেন।

সেলগুলি ক্রম অনুসারে চালান। প্রতিটি অংশ সংক্ষিপ্ত এবং স্বতন্ত্র।


## সেটআপ

দুটি নির্ভরতা ইনস্টল করুন। দুটোরই মুক্ত লাইসেন্স রয়েছে (Apache-2.0 / MIT)।


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## সহায়ক উপযোগিতা

এই দুইটি সহায়ক মূলত base64url এনকোডিং (প্যাডিং ছাড়া) এবং কোনোও অবজেক্টের SHA-256 হ্যাশিং পরিচালনা করে। এগুলি নোটবুকের বাকিটা রশিদের লজিকেই কেন্দ্রীভূত রাখে।


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## বিভাগ ১: আপনার প্রথম রসিদে সাইন করুন

কল্পনা করুন আমাদের এজেন্ট **Contoso Travel** একটি গ্রাহকের জন্য সিডনি থেকে লস অ্যাঞ্জেলস যাওয়ার ফ্লাইট খোঁজেন। আমরা এই টুল কলটি একটি স্বাক্ষরিত রসিদ হিসাবে রেকর্ড করতে চাই যাতে ভবিষ্যতের কোনও অডিটর এটি যাচাই করতে পারে আমাদের উপর বিশ্বাস না করেই।

### ধাপ ১.১: একটি সাইনিং কী তৈরি করুন

উৎপাদন পরিবেশে, এজেন্টের সাইনিং কী সাধারণত একটি হার্ডওয়্যার সুরক্ষা মডিউল (HSM), Azure Key Vault, অথবা সমপর্যায়ের সুরক্ষিত স্টোরে থাকত। এই পাঠে আমরা মেমরিতে একটি নতুন কী তৈরি করব।


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ধাপ ১.২: রিসিপ্ট পে-লোড তৈরি করুন

পে-লোডে সমস্ত কিছু থাকে যেটা আমরা রিসিপ্টে প্রমাণিত করতে চাই: কে কাজ করেছে, কোন টুল ব্যবহার করেছে, কোন আর্গুমেন্টসের সঙ্গে, কী ফলাফল এসেছে, কোন নীতির অধীনে, এবং কখন। আমরা আর্গুমেন্টস এবং ফলাফলকে ইনলাইন না রেখে হ্যাশ করি যাতে রিসিপ্ট সংবেদনশীল বিষয়বস্তু ফাঁস না করে।


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### ধাপ ১.৩: রসিদে স্বাক্ষর করুন এবং সংযোজন করুন

তিনটি ধাপ:

১. JCS ব্যবহার করে পে লোডটি canonicalize করুন যাতে একই যৌক্তিক রসিদ তৈরি করা দুটি ইমপ্লিমেন্টেশন বাইট-একই বাইট তৈরি করে।
২. SHA-256 দিয়ে canonical বাইটগুলোর হ্যাশ করুন।
৩. Ed25519 প্রাইভেট কী দিয়ে হ্যাশে স্বাক্ষর করুন।

তারপর স্বাক্ষরটি মূল পে লোডে সংযুক্ত করা হয় চূড়ান্ত রসিদ তৈরির জন্য।


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ধাপ ১.৪: রসিদ যাচাই করুন

যাচাই প্রক্রিয়াটি বিপরীত। আমরা স্বাক্ষর সরিয়ে ফেলি, canonical হ্যাশ পুনরায় হিসাব করি, এবং রসিদে থাকা পাবলিক কী এর সাথে স্বাক্ষরটি মিলাই।

এই যাচাইটি করার জন্য একজন নিরীক্ষক আমাদের থেকে রসিদ ছাড়া আর কিছুই চায় না। কোনও পরিষেবা কল করতে হবে না, কোনও কী ডিরেক্টরি অনুসন্ধান করতে হবে না, কোনও বিশ্বাস প্রয়োজন নেই।


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

আপনি দেখতে পাবেন `Receipt is valid: True`। এজেন্ট তার প্রথম ক্রিপ্টোগ্রাফিক্যালি স্বাক্ষরিত অডিট রেকর্ড তৈরি করেছে।


## অধ্যায় ২: রশিদে হস্তক্ষেপ করুন

রশিদের পুরো উদ্দেশ্য হল সেগুলো হস্তক্ষেপ-প্রমাণ হওয়া। চলুন এটি প্রমাণ করি।

আমরা রশিদের ঠিক এক অক্ষর পরিবর্তন করব এবং যাচাই ব্যর্থ হতে দেখব।


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### কি নতুন ঘটল?

যখন আমরা `policy_id` পরিবর্তন করলাম, তখন canonical বাইটগুলি পরিবর্তিত হয়। সেই বাইটগুলির SHA-256 হ্যাশ পরিবর্তিত হয়। সিগনেচারটি (যা মূল হ্যাশের উপর ছিল) আর নতুন হ্যাশের সাথে মেলে না। ভেরিফিকেশন সঠিকভাবে `False` রিটার্ন করে।

রশিদটির যেকোনো ক্ষেত্র পরিবর্তন করেও এটি যাচাই করা সম্ভব নয়, যদি না আক্রমণকারী কাছে প্রাইভেট কী থাকে। যতক্ষণ প্রাইভেট কী একটি কী ভল্টে থাকে এবং পাবলিক কী প্রকাশিত থাকে, ছলচাতুরী লুকানো অসম্ভব।

নিজে চেষ্টা করুন: উপরের সেলে `tool_name` বা `agent_id` বা `timestamp` পরিবর্তন করুন এবং পুনরায় চালান। প্রতিটি পরিবর্তন অবৈধ রশিদ তৈরি করে।


## Section 3: রসিদগুলোকে একত্রিত করুন

একটি রসিদ একটি ক্রিয়াকলাপকে সুরক্ষিত রাখে। বেশিরভাগ এজেন্ট অনেকগুলি ক্রিয়াকলাপ গ্রহণ করে। পুরো ক্রমটি টেম্পার-প্রমাণিক করার জন্য, আমরা প্রতিটি রসিদকে আগের রসিদটির সাথে সংযুক্ত করি আগের রসিদটির হ্যাশ নতুন রসিদের পেλόডে অন্তর্ভুক্ত করে।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

যদি কেউ কোনো রসিদ সরিয়ে ফেলে বা পুনর্বিন্যাস করে, চেইন ঠিক সেই বিন্দুতে ভেঙে যায়। পরবর্তী যে কোনো রসিদের যাচাইকরণ ব্যর্থ হয় কারণ তার `previous_receipt_hash` আর তার পূর্ববর্তী রসিদের প্রকৃত হ্যাশের সাথে মেলে না।


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

এখন মধ্যবর্তী রসিদ দিয়ে জালিয়াতি করে চেইন ভেঙে দিন এবং পুনরায় যাচাই করুন। জালিয়াতিপূর্ণ রসিদ তার স্বাক্ষর পরীক্ষায় ব্যর্থ হয়, এবং পরবর্তী রসিদ তার চেইন-লিঙ্ক পরীক্ষায় ব্যর্থ হয় (কারণ এর `previous_receipt_hash` আর পরিবর্তিত মধ্যবর্তী রসিদের হ্যাশের সাথে মিলে না)।


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

রশিদ 0 এখনও যাচাই করে (এটি পরিবর্তন করা হয়নি এবং নির্ভর করার মতো কোনো পূর্বসূরী নেই)। রশিদ 1 এর স্বাক্ষর পরীক্ষা ব্যর্থ হয় কারণ আমরা `tool_args_hash` পরিবর্তন করেছি। রশিদ 2 এর চেইন-লিঙ্ক পরীক্ষা ব্যর্থ হয় কারণ এর `previous_receipt_hash` মূল (এখন পরিবর্তিত) রশিদ 1 এর বিরুদ্ধে গণনা করা হয়েছিল।

এমনকি যদি একজন আক্রমণকারী পরিবর্তিত রশিদ 1 পুনরায় স্বাক্ষর করে (যা তারা ব্যক্তিগত কী ছাড়া করতে পারে না), রশিদ 2 তে চেইন-লিঙ্ক অসমতা এখনও ছলনাপূর্ণতা প্রকাশ করবে। পরিবর্তন লুকানোর জন্য, আক্রমণকারীকে পরিবর্তনের পয়েন্ট থেকে প্রতিটি রশিদ পুনরায় স্বাক্ষর করতে হবে, যা ব্যক্তিগত কী-এর অধিকার দাবি করে।


## বিভাগ ৪: রসিদ স্বাক্ষরের সাথে একটি এজেন্ট টুল কল মোড়ানো

একটি বাস্তব স্থাপনায়, আপনি চান না প্রতিটি এজেন্ট লেখক `make_receipt` কল করতে মনে রাখুক। আপনি চান প্রতিটি টুল আহ্বানের জন্য রসিদ স্বাক্ষর স্বয়ংক্রিয় হোক।

এখানে সবচেয়ে সহজ প্যাটার্নটি হল: একটি র‍্যাপার ক্লাস যা যেকোনো কলযোগ্য টুল ফাংশন নেয় এবং তার একটি রসিদ-প্রদানকারী সংস্করণ প্রদান করে। এটি যেকোনো এজেন্ট ফ্রেমওয়ার্কের সাথে মানিয়ে যায়, যার মধ্যে Microsoft Agent Framework (`agent_framework.foundry`) অন্তর্ভুক্ত।

যদি আপনার Microsoft Foundry প্রকল্প সেট আপ না থাকে, নিচের স্থানীয় মকটি এখনও প্যাটার্নটি প্রদর্শন করে।


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্কের সাথে ইন্টিগ্রেশন

উপরের `ReceiptedTool` র‍্যাপার ফ্রেমওয়ার্ক-নিরপেক্ষ। এটি মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক দিয়ে নির্মিত একটি এজেন্টের ভিতরে ব্যবহার করতে, র‍্যাপ করা ফাংশনটিকে একটি টুল হিসাবে নিবন্ধন করুন। একটি স্কেচ (আপনি মকটি একটি বাস্তব মাইক্রোসফট ফাউন্ড্রি টুল নিবন্ধনের সাথে প্রতিস্থাপন করবেন):

```python
# ইন্টিগ্রেশন আকার দেখানো ছদ্মকোড।
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # মোড়ানো টুল, কাঁচা ফাংশন নয়
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # রান করার পরে, এজেন্ট যে প্রতিটি টুল কল করেছে তার একটি সই করা রসিদ আছে:
# audit_chain = receipted_lookup.receipts
```

এজেন্ট ফ্রেমওয়ার্কের রসিদ সম্পর্কে কিছু জানা দরকার নেই। রসিদ সাইনিং টুলের চারপাশে মোড়ানো হয়, ফ্রেমওয়ার্কের সাথে সরাসরি সংযুক্ত নয়। এটাই কিভাবে আপনি এজেন্ট পুনর্লিখন ছাড়াই বিদ্যমান এজেন্ট কোডে উৎপত্তি যোগ করেন।


## পুনঃস্মরণ এবং চ্যালেঞ্জ বাড়ানো

আপনার আছে:

- একটি Ed25519 কী পেয়ার তৈরি করেছেন।
- একটি এজেন্ট টুল কলের জন্য রসিদ তৈরি এবং স্বাক্ষর করেছেন।
- শুধুমাত্র পাবলিক কী ব্যবহার করে অফলাইনে রসিদ যাচাই করেছেন।
- একটি রসিদে পরিবর্তন করেছেন এবং যাচাই ব্যর্থ হয়েছে দেখতে পেয়েছেন।
- তিনটি রসিদের একটি হ্যাশ-চেইন ক্রম তৈরি করেছেন।
- চেইনের মাঝখানে পরিবর্তন করেছেন এবং স্বাক্ষরের ব্যর্থতা এবং চেইন-লিঙ্কের ব্যর্থতা উভয়কেই পর্যবেক্ষণ করেছেন।
- একটি টুল ফাংশনকে স্বয়ংক্রিয় রসিদ স্বাক্ষরসহ আবৃত করেছেন।

**চ্যালেঞ্জ বাড়ানো।** রসিদ স্কিমায় একটি `request_id` ফিল্ড যোগ করুন (বিতরণ করা ট্রেসিংয়ের জন্য UUID)। `make_receipt`-কে এটি অন্তর্ভুক্ত করার জন্য আপডেট করুন, এবং নিশ্চিত করুন রসিদগুলি এখনও সম্পূর্ণ ভাবে যাচাই করা যায়। তারপর স্বাক্ষর করার পর ফিল্ডটি পরিবর্তন করুন এবং যাচাই ব্যর্থ হয় কিনা দেখুন। এটি আপনাকে শেখায় কীভাবে ক্যাননিক্যাল এনকোডিংয়ের প্রতিটি বাইট স্বাক্ষরের সাথে সম্পর্কিত।

**গুরুত্বপূর্ণ সীমানা।** রসিদ তিনটি ব্যাপার প্রমাণ করে এবং শুধুমাত্র তিনটি: স্বত্বা (এই কী এই বিষয়বস্তু স্বাক্ষর করেছে), অখণ্ডতা (স্বাক্ষরের পর থেকে বিষয়বস্তু পরিবর্তিত হয়নি), এবং ক্রম (এই রসিদটি ঐ রসিদের পর এসেছে)। তারা প্রমাণ করে না যে এজেন্টের কাজ সঠিক ছিল, `policy_id`-তে যে নীতিমালা উল্লেখ ছিল তা কার্যকর হয়েছিল, বা এজেন্ট সব নিয়ম পালন করেছে। রসিদ হলো একটি ভিত্তি। গভর্নেন্স হলো আপনি যে সিস্টেম তৈরি করবেন তার উপরে।

এই সীমানাটি মাথায় রেখে আবার পাঠের README পড়ুন। দলের সবচেয়ে সাধারণ ভুল হলো রসিদ থাকার মানে "আমরা গভর্নরড" এটা ভাবা। তা নয়। রসিদ এজেন্টের আচরণ অডিটযোগ্য করে তোলে। তারা এটিকে সঠিক করে না।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
